In [33]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain langchain-community
%pip install -Uq python_dotenv
%pip install -Uq sentence-transformers
%pip install -Uq supabase

You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart t

In [34]:
from supabase import create_client
from dotenv import load_dotenv
import os
load_dotenv()

SUPABASE_URL=os.getenv("SUPABASE_URL")
SUPABASE_KEY=os.getenv("SUPABASE_KEY")

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [35]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_community.vectorstores import SupabaseVectorStore


In [36]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# # Test with your PDF file
# file_path = "./docs/constitution.pdf"  # Change this to your PDF path
# elements = partition_document(file_path)

In [37]:
# elements

In [38]:
# All types of different atomic elements we see from unstructured
# set([str(type(el)) for el in elements])

In [39]:
# len(elements)

In [40]:
# elements[0].to_dict()

In [41]:
# Gather all images
# images = [element for element in elements if element.category == 'Image']
# print(f"Found {len(images)} images")

# images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

In [42]:
# Gather all table
# tables = [element for element in elements if element.category == 'Table']
# print(f"Found {len(tables)} tables")

# tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

In [43]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
# chunks = create_chunks_by_title(elements)

In [44]:
# View all chunks
# chunks

# # All unique types
# set([str(type(chunk)) for chunk in chunks])

In [45]:
# View a single chunk
# chunks[5].to_dict()

# View original elements
# chunks[11].metadata.orig_elements[-1].to_dict()

In [46]:
# here we have a chunk with text and a table. We want to be able to identify both the text and the table(html), and store them in a structured way so we can use them later for retrieval and question answering.

def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data



In [47]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOllama(model="mistral",temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [48]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [49]:
# Process chunks with AI
# processed_chunks = summarise_chunks(chunks)

In [ ]:
# processed_chunks

In [51]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
# json_data = export_chunks_to_json(processed_chunks)

In [ ]:
def create_vector_store(documents):
    print("🔮 Creating embeddings and storing in pgvector...")

    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    vectorstore = SupabaseVectorStore.from_documents(
        documents=documents,
        embedding=embedding_model,
        client=supabase,
        table_name="documents",
        query_name="match_documents"
    )

    print("✅ Vector store created")

    return vectorstore

# db = create_vector_store(processed_chunks)

In [53]:
def retrieve_from_supabase(query: str, k: int = 3):
    """Direct Supabase RPC retrieval — bypasses SupabaseVectorStore which is broken with supabase>=2.0"""
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    query_embedding = embedding_model.embed_query(query)

    result = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_count": k,
            "filter": {},
        }
    ).execute()

    return [
        Document(
            page_content=row["content"],
            metadata=row.get("metadata", {})
        )
        for row in result.data
    ]

query = "What is 18th amendment?"
chunks = retrieve_from_supabase(query, k=5)

export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 5 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': 'PART XI\n\nAmendment of Constitution\n\n238. Amendment of Constitution\n\n238. Subject to this Part, the Constitution may be amended by Act of 1[Majlis-e-Shoora (Parliament)].\n\n239. Constitution, amendment Bill\n\n2[239. (1) A Bill to amend the Constitution may originate in either House and, when the Bill has been passed by the votes of not less than two-thirds of the total membership of the House, it shall be transmitted to the other House.\n\n(2) If the Bill is passed without amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall, subject to the provisions of clause (4), be presented to the President for assent.\n\n(3) If the Bill is passed with amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall be reconsidered by the House in which it had originated, and if the Bill 

In [54]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks)
    
    print("🎉 Pipeline completed successfully!")
    return db

In [55]:
# Run the complete pipeline
db = run_complete_ingestion_pipeline("./docs/constitution.pdf")

The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: ./docs/constitution.pdf


The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


✅ Extracted 3845 elements
🔨 Creating smart chunks...
✅ Created 245 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/245
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     ❌ AI summary failed: Ollama call failed with status code 400. Details: {"error":"{\"error\":{\"code\":400,\"message\":\"Multimodal data provided, but model does not support multimodal requests.\",\"type\":\"invalid_request_error\"}}"}
     → AI summary created successfully
     → Enhanced content preview: THE CONSTITUTION OF THE ISLAMIC REPUBLIC OF PAKISTAN

[As modified upto the 28th February, 2012]

NATIONAL ASSEMBLY OF PAKISTAN

PREFACE

The National Assembly of Pakistan passed the Constitution on 1...
   Processing chunk 2/245
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/245
     Types found: ['table', 'text']
     Tables: 1, Images: 0
     → Creating AI summar

# RETREIVAL PIPELINE

In [56]:
%pip install langchain_cohere
%pip install rank_bm25


You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [57]:
from langchain.retrievers import EnsembleRetriever, BM25Retriever
from langchain_cohere import CohereRerank

#### SETUP: Create the three types of retrievers

# 1. Vector Retriever (Semantic Search/Dense Retrieval)
Setting up hybrid retriever...

In [58]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun

class SupabaseRetriever(BaseRetriever):
    """Custom retriever using direct RPC — compatible with supabase>=2.0"""
    k: int = 15

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        return retrieve_from_supabase(query, k=self.k)

vector_retriever = SupabaseRetriever(k=15)

# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [59]:
# 2. BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(processed_chunks)
bm25_retriever.k = 15

# 3. Hybrid Retriever

In [60]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

print("Setup complete!\n")

Setup complete!



## Step 1: Get results from hybrid search (retrieve more chunks)

In [61]:
query = "What is the 18th amendment?"

print("STEP 1: Hybrid Search Results")
print("-"*50)

retrieved_docs = hybrid_retriever.invoke(query)  # Get top 25 for reranking

# Show top 10 from hybrid search
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print(f"\n(Retrieved {len(retrieved_docs)} total chunks for reranking)\n")

STEP 1: Hybrid Search Results
--------------------------------------------------
 1. PART XI

Amendment of Constitution

238. Amendment of Constitution

238. Subject to this Part, the Constitution may be amended by Act of 1[Majlis-e-Shoora (Parliament)].

239. Constitution, amendment Bill

2[239. (1) A Bill to amend the Constitution may originate in either House and, when the Bill has been passed by the votes of not less than two-thirds of the total membership of the House, it shall be transmitted to the other House.

(2) If the Bill is passed without amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall, subject to the provisions of clause (4), be presented to the President for assent.

(3) If the Bill is passed with amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall be reconsidered by the House in which it h

## Step 2: Apply Cohere Reranking

In [62]:
print("STEP 2: After Cohere Reranking (Top 10)")
print("-"*50)

# Initialize Cohere reranker
reranker = CohereRerank(model="rerank-english-v3.0", top_n=10)

# Rerank the retrieved documents
reranked_docs = reranker.compress_documents(retrieved_docs, query)

# Show reranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print("\n" + "="*80)
print("ANALYSIS:")
print("✅ Hybrid Search: Mixed relevant and irrelevant results")
print("✅ Reranking: Most relevant Tesla financial/production info at top")
print("✅ Notice how reranking moved the most contextually relevant chunks higher")

# Optional: Show the difference more clearly
print("\n" + "="*80)
print("KEY IMPROVEMENTS AFTER RERANKING:")
print("-"*40)


hybrid_top_5 = [doc.page_content for doc in retrieved_docs[:5]]
reranked_top_5= [doc.page_content for doc in reranked_docs[:5]]

print("BEFORE (Hybrid Top 3):")
for i, content in enumerate(hybrid_top_5, 1):
    print(f"  {i}. {content}")

print("\nAFTER (Reranked Top 3):")
for i, content in enumerate(reranked_top_5, 1):
    print(f"  {i}. {content}")

STEP 2: After Cohere Reranking (Top 10)
--------------------------------------------------
 1. (b) such law or provision shall, to the extent to which it is held to be so repugnant, cease to have effect on the day on which the decision of the Court takes effect.

7*

*

*

*

*

*

*

*

*

1 The words ―or the Concurrent Legislative List‖ omitted by the Constitution (Eighteenth Amdt.) Act, 2010 (10 of 2010), s. 75.

2 Subs. ibid., for the words ―in the either of those lists‖.

3 Subs. and shall be deemed always to have been so subs. by the Constitution (Amdt.) order, 1984 (P.O. No. 1 of 1984), Art. 2, for the full stop.

4 Proviso added and shall be deemed always to have been so added ibid.

5 The words ―or the Concurrent Legislative List‖ stand omitted as consequence of the (Eighteenth Amdt.) Act, 2010 (10 of 2010), see section 2.

6 Subs. ibid., for ―Either of those Lists‖.

7 Clause (4) omitted by the Constitution (Second Amdt.) Order, 1980 (P. O. No. 4 of 1980), Art. 3.

112

CONST

In [63]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.chat_models import ChatOllama

print("\n" + "="*80)
print("FINAL: RAG with Reranked Context")
print("-"*40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents."""

# Setup AI model
model = ChatOllama(model="mistral")
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = model.invoke(messages)
print("Generated Response:")
print(result.content)


FINAL: RAG with Reranked Context
----------------------------------------
Generated Response:
 The 18th Amendment refers to the Constitution (Eighteenth Amendment) Act, 2010 (10 of 2010) in the context of the Constitution of Pakistan. This act introduced new Articles 267A and 267B into the constitution. However, specific details about what these articles entail are not provided in the given documents. The act also pertains to the continuance of existing laws, the President's powers to adapt these laws within a period of two years from the commencing day, and court interpretations of existing laws.
